In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/talishanand/hr-policy-pdfs/10_Travel_and_Expense_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/03_Work_From_Home_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/09_Onboarding_and_Separation_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/07_IT_and_Data_Security_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/06_Compensation_and_Benefits_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/app.py
/kaggle/input/datasets/talishanand/hr-policy-pdfs/04_Code_of_Conduct.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/01_Employee_Handbook.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/05_Performance_Review_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/02_Leave_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/08_Prevention_of_Sexual_Harassment_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/00_Company_Profile.pdf


# Zyro Dynamics HR Help Desk — RAG Challenge
### NxtWave Masterclass | Build an HR chatbot using RAG

---

## Objective

Build a Retrieval-Augmented Generation (RAG) pipeline that answers employee HR questions using internal policy documents.

## What you will build

- Load and process HR policy documents
- Create chunks and embeddings
- Build a vector database using FAISS
- Implement a RAG pipeline with guardrails
- Deploy a Streamlit chatbot
- Generate your `submission.csv`

## Submission Requirements

1. `submission.csv` — upload on Kaggle
2. Streamlit App URL
3. LangSmith Trace URL

---

> Follow the notebook cells sequentially and complete the sections marked for implementation.

## Cell 1 — Install Dependencies

> ⚠️ Run this cell first before anything else.

This cell installs all required libraries for:
- document loading
- embeddings
- vector database
- RAG pipeline
- Streamlit deployment
- LangSmith tracing

> After installation completes, restart the kernel/runtime and run all cells from the top.

In [2]:
print("Installing required packages...\n")

!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    langchain-google-genai \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface_hub \
    groq \
    streamlit \
    langsmith \
    python-dotenv \
    tiktoken

print("\nInstallation complete.")
print("Please restart the kernel/runtime before running the next cell.")

Installing required packages...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.2/552.2 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

## Cell 2 — Configuration

This is the main configuration cell for the notebook.

Here you can:
- choose your LLM provider
- select the model you want to use
- update related settings if needed

All remaining cells will automatically use this configuration.

In [3]:
LLM_PROVIDER = "groq"           # "groq" | "gemini" | "openai"
LLM_MODEL    = "llama-3.1-8b-instant"  # fast & free on Groq

CORPUS_PATH = "/kaggle/input/datasets/talishanand/hr-policy-pdfs"
# this is the folder that directly contains 00_Company_Profile.pdf, 01_Employee_Handbook.pdf, ...

print(f"Provider: {LLM_PROVIDER}")
print(f"Model: {LLM_MODEL}")

Provider: groq
Model: llama-3.1-8b-instant


## Cell 3 — Imports

This cell imports all required libraries for:
- document loading
- text chunking
- embeddings
- vector search
- prompt handling
- LangSmith tracing

> Run this cell without modifying anything.

In [4]:
import os, json, time, csv
from cryptography.fernet import Fernet
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langsmith import traceable

print("Imports loaded successfully.")

/tmp/ipykernel_23/3671248947.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Imports loaded successfully.


## Cell 4 — API Keys + LangSmith Setup

In [5]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()

    if LLM_PROVIDER == "groq":
        os.environ["GROQ_API_KEY"] = secrets.get_secret("GROQ_API_KEY")
    elif LLM_PROVIDER == "gemini":
        os.environ["GOOGLE_API_KEY"] = secrets.get_secret("GOOGLE_API_KEY")
    elif LLM_PROVIDER == "openai":
        os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")

    os.environ["LANGCHAIN_API_KEY"]    = secrets.get_secret("LANGCHAIN_API_KEY")
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = "zyro-rag-challenge"
    print("Running on Kaggle — secrets loaded!")

except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = "zyro-rag-challenge"

SUBMISSION_SECRET = b"6Q_EBPtBG-60URcrF6jxNTJSRjy-CtZbJlvp_xf0c_M="
fernet = Fernet(SUBMISSION_SECRET)

print("Environment configured successfully.")

Running on Kaggle — secrets loaded!
Environment configured successfully.


## Cell 5 — Load Documents

Load all policy PDFs from the corpus directory.

In [6]:
# Initialize document loader
loader = PyPDFDirectoryLoader(CORPUS_PATH)

# Load documents
documents = loader.load()

print(f"Loaded {len(documents)} documents")

Loaded 39 documents


In [7]:
import os

print("Listing files under /kaggle/input:")
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Listing files under /kaggle/input:
/kaggle/input/datasets/talishanand/hr-policy-pdfs/10_Travel_and_Expense_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/03_Work_From_Home_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/09_Onboarding_and_Separation_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/07_IT_and_Data_Security_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/06_Compensation_and_Benefits_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/app.py
/kaggle/input/datasets/talishanand/hr-policy-pdfs/04_Code_of_Conduct.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/01_Employee_Handbook.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/05_Performance_Review_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/02_Leave_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/08_Prevention_of_Sexual_Harassment_Policy.pdf
/kaggle/input/datasets/talishanand/hr-policy-pdfs/00_Company_Profile.pdf


## Cell 6 — Chunk Documents

Split documents into overlapping chunks for better retrieval.

In [8]:
# Initialize text splitter with overlap to preserve context across chunk boundaries
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Create document chunks
chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

Created 112 chunks


## Cell 7 — Embeddings

Initialize a sentence-transformer embedding model.

In [9]:
# Use a strong, lightweight embedding model well-suited for semantic search
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model initialized.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model initialized.


## Cell 8 — Vector Store + Retriever

Build FAISS vector store and create an MMR retriever for diverse, relevant results.

In [10]:
# Build FAISS vector store from document chunks
vectorstore = FAISS.from_documents(chunks, embeddings)

# Create MMR retriever: balances relevance + diversity to surface the best chunks
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,           # return top-5 chunks
        "fetch_k": 20,    # consider top-20 candidates before MMR re-ranking
        "lambda_mult": 0.7  # 0 = max diversity, 1 = max relevance
    }
)

print("Vector store initialized.")

Vector store initialized.


## Cell 9 — LLM Initialization

The language model is initialized using the configuration from Cell 2.

In [11]:
if LLM_PROVIDER == "groq":
    from langchain_groq import ChatGroq
    llm = ChatGroq(
        model=LLM_MODEL,
        temperature=0.1,
        max_tokens=512
    )

elif LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(
        model=LLM_MODEL,
        temperature=0.1,
        max_output_tokens=512
    )

elif LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        model=LLM_MODEL,
        temperature=0.1,
        max_tokens=512
    )

else:
    raise ValueError("Unsupported LLM provider.")

print("LLM initialized.")

LLM initialized.


## Cell 10 — Build the RAG Chain

Implement the RAG pipeline using LCEL with LangSmith tracing.

In [12]:
# System prompt: instructs the LLM to answer strictly from retrieved context
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     """You are an HR Help Desk assistant for Zyro Dynamics Pvt. Ltd.
Your job is to answer employee HR questions accurately and concisely using ONLY the information
provided in the context below. Do not use any prior knowledge or make up information.

Rules:
- Answer ONLY from the provided context.
- If the context does not contain enough information to answer, say so clearly.
- Be concise but complete. Use bullet points for lists.
- Always cite the relevant policy document name when possible.

Context:
{context}"""),
    ("human", "{question}")
])


def format_docs(docs):
    """Concatenate retrieved document chunks with source metadata."""
    return "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'Unknown')}, Page: {doc.metadata.get('page', '?')}]\n{doc.page_content}"
        for doc in docs
    )


@traceable(name="rag_chain")  # LangSmith tracing decorator
def rag_chain(question: str):
    """Retrieve relevant chunks and generate a grounded answer."""
    # Retrieve relevant documents
    docs = retriever.invoke(question)
    
    # Format documents as context string
    context = format_docs(docs)
    
    # Build prompt and invoke LLM
    prompt_value = RAG_PROMPT.invoke({"context": context, "question": question})
    answer = llm.invoke(prompt_value)
    answer_text = StrOutputParser().invoke(answer)
    
    # Return answer + sources for citation display
    sources = list({
        doc.metadata.get("source", "Unknown") for doc in docs
    })
    
    return {"answer": answer_text, "sources": sources, "docs": docs}


print("RAG pipeline initialized.")

RAG pipeline initialized.


## Cell 11 — Guardrails

Detect and refuse out-of-scope questions before routing to the RAG pipeline.

In [13]:
# Guardrail prompt: classifies whether a question is HR-related or out-of-scope
OOS_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     """You are a classifier that decides whether a user question is related to HR policies
at Zyro Dynamics Pvt. Ltd.

HR-related topics include: leave, salary, payroll, benefits, performance reviews, promotions,
work from home, attendance, onboarding, resignation, termination, code of conduct, POSH,
travel reimbursements, IT security policy, employee grade structure, compensation, insurance,
provident fund, gratuity, or any other topic covered in company HR policy documents.

Respond with ONLY one word:
- "IN_SCOPE" if the question is about Zyro Dynamics HR policies or employee matters.
- "OUT_OF_SCOPE" if the question is about anything else (general knowledge, coding, news, sports, personal advice, etc.)."""),
    ("human", "{question}")
])

# Polite refusal message for out-of-scope questions
REFUSAL_MESSAGE = (
    "I'm sorry, but I can only answer HR-related questions based on Zyro Dynamics' "
    "internal policy documents. Your question appears to be outside my scope. "
    "For other queries, please reach out to the appropriate team."
)


@traceable(name="ask_bot")  # LangSmith tracing decorator
def ask_bot(question: str) -> dict:
    """
    Main chatbot entry point.
    1. Classifies the question as in-scope or out-of-scope.
    2. Routes in-scope questions through the RAG pipeline.
    3. Returns a polite refusal for out-of-scope questions.
    """
    # Step 1: Guardrail classification
    classifier_prompt = OOS_PROMPT.invoke({"question": question})
    classification = llm.invoke(classifier_prompt)
    classification_text = StrOutputParser().invoke(classification).strip().upper()
    
    # Step 2: Route accordingly
    if "OUT_OF_SCOPE" in classification_text:
        return {
            "answer": REFUSAL_MESSAGE,
            "sources": [],
            "is_out_of_scope": True
        }
    
    # Step 3: In-scope → run RAG chain
    result = rag_chain(question)
    result["is_out_of_scope"] = False
    return result


print("Guardrails initialized.")

Guardrails initialized.


## Cell 12 — Test the Bot

Run sample in-scope and out-of-scope questions to validate the pipeline.

In [14]:
test_questions = [
    # In-scope HR questions
    "How many days of earned leave can I carry forward at the end of the financial year?",
    "What is the notice period for an L4 employee who wants to resign?",
    "Am I eligible for Work From Home if I am on probation?",
    # Out-of-scope questions
    "What is the capital of France?",
    "Can you write me a Python script to sort a list?"
]

for i, q in enumerate(test_questions, 1):
    print(f"Q{i}: {q}")
    result = ask_bot(q)
    print(f"Answer: {result['answer']}")
    if result['sources']:
        print(f"Sources: {', '.join(result['sources'])}")
    print("-" * 60)

Q1: How many days of earned leave can I carry forward at the end of the financial year?
Answer: According to the Leave Policy document (Page 1), Casual Leave and Sick Leave cannot be carried forward to the next financial year, nor can they be encashed. However, it does not explicitly state the carry forward limit for Earned Leave.

To find the answer, we need to refer to the Eligibility and Accrual section of the Earned Leave policy. Unfortunately, it does not mention the carry forward limit for Earned Leave either.

However, we can infer that Earned Leave can be carried forward, as it is not mentioned that it lapses automatically at the end of the financial year like Casual Leave and Sick Leave.
Sources: /kaggle/input/datasets/talishanand/hr-policy-pdfs/09_Onboarding_and_Separation_Policy.pdf, /kaggle/input/datasets/talishanand/hr-policy-pdfs/02_Leave_Policy.pdf, /kaggle/input/datasets/talishanand/hr-policy-pdfs/10_Travel_and_Expense_Policy.pdf
----------------------------------------

## Cell 13 — LangSmith Trace URL

Get your shareable LangSmith trace URL for submission.

In [15]:
print("""
HOW TO GET YOUR LANGSMITH TRACE URL
════════════════════════════════════
1. Go to: https://smith.langchain.com
2. Sign in with your account
3. Click on project: 'zyro-rag-challenge'
4. You will see all your RAG traces here
5. Top right corner → Share → Enable Public Link
6. Copy the URL
7. Paste this URL when running Cell 16!
""")


HOW TO GET YOUR LANGSMITH TRACE URL
════════════════════════════════════
1. Go to: https://smith.langchain.com
2. Sign in with your account
3. Click on project: 'zyro-rag-challenge'
4. You will see all your RAG traces here
5. Top right corner → Share → Enable Public Link
6. Copy the URL
7. Paste this URL when running Cell 16!



## Cell 14 — Streamlit App

Build and save the Streamlit chatbot application as `app.py`.

This cell writes the app code to disk — deploy it on https://share.streamlit.io

In [16]:
app_code = '''
import os
import streamlit as st
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

# ── Page Config ────────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Zyro Dynamics HR Help Desk",
    page_icon="🏢",
    layout="centered"
)

# ── Constants ──────────────────────────────────────────────────────────────────
CORPUS_PATH = "hr_docs/"   # place your 11 PDFs in this folder before deploying
REFUSAL_MESSAGE = (
    "I\'m sorry, I can only answer HR-related questions from Zyro Dynamics\' "
    "internal policy documents. Your question appears to be outside my scope. "
    "For other matters, please contact the appropriate team."
)

# ── Build RAG pipeline (cached so it runs only once per session) ───────────────
@st.cache_resource(show_spinner="Loading HR policy documents...")
def build_pipeline():
    # 1. Load PDFs
    loader = PyPDFDirectoryLoader(CORPUS_PATH)
    documents = loader.load()

    # 2. Chunk
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800, chunk_overlap=150,
        separators=["\\n\\n", "\\n", ". ", " ", ""]
    )
    chunks = splitter.split_documents(documents)

    # 3. Embed
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )

    # 4. Vector store with MMR retriever
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.7}
    )

    # 5. LLM
    llm = ChatGroq(
        api_key=os.environ["GROQ_API_KEY"],
        model="llama-3.1-8b-instant",
        temperature=0.1,
        max_tokens=512
    )

    return retriever, llm


def format_docs(docs):
    return "\\n\\n".join(
        f"[Source: {d.metadata.get(\'source\', \'Unknown\')}, Page {d.metadata.get(\'page\', \'?\')}]\\n{d.page_content}"
        for d in docs
    )


RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     """You are an HR Help Desk assistant for Zyro Dynamics Pvt. Ltd.
Answer employee questions ONLY using the context below. Be concise and cite the policy document.
If the context does not contain sufficient information, say so clearly.

Context:
{context}"""),
    ("human", "{question}")
])

OOS_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     """Classify the user question as HR-related or not.
HR topics: leave, salary, payroll, benefits, performance, WFH, attendance, onboarding,
resignation, code of conduct, POSH, travel reimbursements, IT policy, compensation, insurance.
Reply ONLY with IN_SCOPE or OUT_OF_SCOPE."""),
    ("human", "{question}")
])


def ask_bot(question, retriever, llm):
    # Guardrail
    cls_prompt = OOS_PROMPT.invoke({"question": question})
    cls = StrOutputParser().invoke(llm.invoke(cls_prompt)).strip().upper()
    if "OUT_OF_SCOPE" in cls:
        return {"answer": REFUSAL_MESSAGE, "sources": [], "is_oos": True}

    # RAG
    docs = retriever.invoke(question)
    context = format_docs(docs)
    prompt_val = RAG_PROMPT.invoke({"context": context, "question": question})
    answer = StrOutputParser().invoke(llm.invoke(prompt_val))
    sources = sorted({d.metadata.get("source", "Unknown") for d in docs})
    return {"answer": answer, "sources": sources, "is_oos": False}


# ── UI ─────────────────────────────────────────────────────────────────────────
st.title("🏢 Zyro Dynamics HR Help Desk")
st.caption("Ask any question about Zyro Dynamics HR policies — leave, salary, performance, WFH, and more.")
st.divider()

# Load pipeline
retriever, llm = build_pipeline()

# Chat history
if "messages" not in st.session_state:
    st.session_state.messages = []

# Render chat history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if msg.get("sources"):
            with st.expander("📄 Sources"):
                for s in msg["sources"]:
                    st.markdown(f"- `{s}`")

# Chat input
if prompt := st.chat_input("Ask an HR question..."):
    # Show user message
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Generate response
    with st.chat_message("assistant"):
        with st.spinner("Looking up HR policies..."):
            result = ask_bot(prompt, retriever, llm)

        st.markdown(result["answer"])

        if result["sources"]:
            with st.expander("📄 Sources"):
                for s in result["sources"]:
                    st.markdown(f"- `{s}`")

    st.session_state.messages.append({
        "role": "assistant",
        "content": result["answer"],
        "sources": result["sources"]
    })

# Sidebar
with st.sidebar:
    st.header("About")
    st.markdown(
        "This chatbot answers HR policy questions for **Zyro Dynamics Pvt. Ltd.** "
        "using Retrieval-Augmented Generation (RAG) over 11 internal policy documents."
    )
    st.divider()
    st.markdown("**Policies covered:**")
    policies = [
        "Company Profile", "Employee Handbook", "Leave Policy",
        "Work From Home Policy", "Code of Conduct", "Performance Review",
        "Compensation & Benefits", "IT & Data Security",
        "POSH Policy", "Onboarding & Separation", "Travel & Expense"
    ]
    for p in policies:
        st.markdown(f"- {p}")

    if st.button("Clear Chat"):
        st.session_state.messages = []
        st.rerun()
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code.strip())

print("app.py created successfully.")

app.py created successfully.


## Cell 15 — Evaluation

Evaluation inputs are loaded automatically at runtime.

> Do not modify this cell.

In [17]:
_Q = [
    ("Q01", "gAAAAABqE-m-EnBhR94RLAsyCD5YUOimCgpyxnGmrg3N29dvcCChh_LbQzGhacqtB6Rg9ySTN-aO4eS5nnSSqgvslxWg3T2XNxvKRw9BoZOGB8sSrPpeXOqPKhdprAkvepa0Ef13rK84Lx_QKNPq5AMeO2zweDFo-UGpOZ1yFV_k0NbpkP0MshR9BpjCI4QpkDSx9QH95aeCK8sqSIkcM8wOFRs1hRD_tV-Jg4XmeHLm4jW6wpCWQRBF-XWIHTwCE3Tod-Zfj-nIFpPe3sNmXFDNY_L5g8aAiw=="),
    ("Q02", "gAAAAABqE-m-iGIUkxaPu-TWqkoQqfrY1QvCn-VC445z8EzeRjBVVSjcBgTYC-OS2QVoM37Oh8tFkJdLJcdivCIg9-jTJ72Vy24BQwagKYrIJlkNBr9yectRVtDZ_X24PWpsbIdMcelH1a6VBz9XXmJ19-0HvqFT0kTeEQEyjzKL2BmtoSHOquqe74xGFhpWD-fI1Cshfxk9EXwgA4poqi7JJ3ovja5pVM18uwfNAmcNacnQRtFTAm6x1JmXKSYVeBSbgpOv1zjEEC-0XfVhF0Wtwli0hRZHhA=="),
    ("Q03", "gAAAAABqE-m-qhjI3OCH68smnD4afuA_GmeOO8rI6R79iaPeodfwbt4NTlWhlbSfgr8BP9ZNAi5yczk65fgsIgbRXQ9AkAVDE2NOD11Aqt6U_NqURkjBQpzn3gzTQNj2qNwtkhx71-l8uYIfZLu8Z-Nv4aAkEaFTKCDp4DWgCaFJbe90TCA2fGUVnDiaI1_0ID87AHR-yYRwTaKYiWI7PiCQWFVm22NGx3cwX_uvMouAEXLX2sw_o3s="),
    ("Q04", "gAAAAABqE-m-qVKLekYizIYVBejJAmZYhT0zftdQzC0nbFt6BAJM52tiRsM0y5pcEfTl7y2bKwjFBSBwj3ik1P1yPTz6mP2h1xHEWoeJxPHdvujlZXJv8ObZO0PbHSPMk6xtnEmEqPAfPLzxjOzu63P3K_0eFdpgR48fUbcQwZt7yZkGzOPqYuUDAE7CBmvgvwRfwymkEzTD8ESt0vYvZdmoYjV7sbScmhoxYbWmjMatFvOzha6D1YA="),
    ("Q05", "gAAAAABqE-m-KRbrY2MpEseeszU46iQWHzbzwOO5-t10vHJrdQOKeaVwPxyp9kiBDCS1Fa5MJyQoTOp2pdEtw9LtUbCEJ_56caOBjtBgngLz4kvcodhVECBLBuD6vsCaQlopu0SardsvA3slA379M8nrcyuuea3dJ97FPlOdQs2b70BRPyOkyNH0nKGqBwQzBlAW7B-ucZwf9dDPPAw-xUTfR3ekIqXReQ=="),
    ("Q06", "gAAAAABqE-m-EYfgWBpxkb_5hGOvvBsAdBu5367Nd5d4uT_6EEAaTeCidG99u5XJ5vcZatZpoj5RjmfrY5O1XNObuApuq_ZFah_StEcLHB31Ow6WRrZpikDGUFJkC-ZfY0TggJzDFvdtwQsIttqNW5js0LMS-74V-AUx0UCi4bABm1vOMGBKP2qGyGTfyh2wfETTw4nNhbac"),
    ("Q07", "gAAAAABqE-m-cZLyG6To-HyWWdEYu42VgbV9c_SCWXt4qJE02YrOFvfMntuBTf-CVXt3MhJWFzrukGMR0-Brla1QMVbefRelzpJqkY2TsIQ3Tcc5MZ0BH6ornHjZAnOd9Iozf1f755EC8hBase1XtbhThrKgYJRKWPxaxKd-nkLK3XuabtmEF8r0bZtTyKVjYNBUWPT--lKJb-pXvw3p3zJ0z6utBLWicmBhgdJvGMoOQCsCLrxi6jrtHZzka7Me7Vm6UUhwSkdz"),
    ("Q08", "gAAAAABqE-m-sxXijCcjguEWTh7qgKt7BX4cbUfFdUwAz6VqSoU4fTnYXUhf-dVQdCKa1lhgc7ZZatU5Pu9iuQHG-ApZCOw2yR-PkZnuY9L7uR02CCJoWYhFQelqYEWYA5uONridoCzD8kh2yqwUSVInEFfBuB2cYgyPobRnP_yRvtaFtLakrMy0fsCZH_zfyrOMVkdF5GoHdPu67XzoEj806x4aS8DJ4ysYFuwNb9zkhhceq_CsU08="),
    ("Q09", "gAAAAABqE-m-nDGYgCF3fSWs2tM39pdnsBua61Ht1ruTZ_NOUmju6AxbGU6WB8HzLEHKQkkCnxc4ka2DohiUSLwVDrWG2ZnGggyt7OnI6D43ovjDBsMhW2jQPaz9zaHua25abfEqF4V1ZioQrdL7lz3D0qzDsjXl4Kw5RY2g3kaDakb62Cb6Dt8badoS-t4Bd_fEAp49t09FH_qwLp_ZTotiFsKFy6QADA=="),
    ("Q10", "gAAAAABqE-m-PwoVsLjWO4nbO8W_65P-UNNF7SjdNZL4sRN-G72eHygPuGyggXwVG8G7HJ2ZmrtCYuNg-rtWH_iuyexPQLVG0EqKr0ZQswJox4iauvFf014qlqr5vC_TtdwHGcMiZsyWZpJauDTffKDm_QJHrGElPUUunCFgX8356s1yMocleGXUBfcZ8B73A5LIALAXRIBpKyt707qYlLhwOG1vhsdR74q21NS0-n0skLZIy7z0pLM="),
    ("Q11", "gAAAAABqE-m-1BAGkhsZEDnkbSwAAwusmnMKdn2gvIM5tltaZ1W-eoKtvbPNu8rkAlOOiOW-9_NobJqDFKDO3J7zCPwWuEdGxwgYpX5sxh2Rg4ngR5R5WDnQsQTPIRHXJkkaN1ufNhvbQ-XOn2Z1QPci8118ByVpkAR5kZTUXOFIZ1IgHP2hbvO4E81GB9CTs9HiZvHAsAnS"),
    ("Q12", "gAAAAABqE-m-NrwI-KspXny9JlQqBEW_eB9jE6bGmnin6IX6SdcB9ol1gR7CmzczDKE6A7XHDOJW20tVHAlGFw-q-J6cWrTajK_mJTv00aHllSozrKiThojuxxnSjhgOhgtNKU5mh7zoz2d2uLo7p-Kl32m4IU6PRsm0kZceID-ZH5ZRw7w4h1qSZOufZO2HvKkR9LtfCQXk"),
    ("Q13", "gAAAAABqE-m-Xr56G8qaFfk3BIVQeDzP5mpahd7wZQ5vGR11AN_sxU1ZzjoPfbSdLmrrhFHEI8S8KhXfjOWZQoMJToWSsnhjZQdrRj0wujH38p2VOZLqqZYSmOflVEQm29z9pAXx_iltLWZLNGf8QsMtZWuo-3SsWt6R2mGvOMBTDj5hCzaq842_r1eupRQJJ1dnTSmNPskW"),
    ("Q14", "gAAAAABqE-m--oxJAL26EQ6bMS5vmgI0pWMWjgbG49qNZu8K_pIiDrp3ro1YFlVvBXOOJ6bSpI7lxz-OXmNrVFkSfJlVf4PchVKfWdddKVT85AMxUHo3PYD15IGV476RznHCiD59twp7x_E6HOF7AFUGiWcsO9Ph63Tfcvh3aJzF7Hk_NPEHcIaaEU9ki2eccYXehJJ3tkmr"),
    ("Q15", "gAAAAABqE-m-3JNAfb2dmCF-2XlNe-F1AaeXybgSJ4DwHtn9o52TEryPYgu-6m70Ivn7izeLy4h44AVbHL_3cv-MWfAwFYp7ct3lvF7dL1QbmhntyeY4c7l0CVPsc-mv8WuY04tpB2XPtHE_0ytl9tQlqAGonC2esnpMbSzgvZPdSw9eHnm5k2Jkh0FbgjLKNWxjdX3Uv2aYDiqOeLMQKZsMMteZzJcwHQ=="),
]

eval_questions = [
    {"question_id": qid, "question": fernet.decrypt(enc.encode()).decode()}
    for qid, enc in _Q
]

print(f"{len(eval_questions)} evaluation questions loaded.")

15 evaluation questions loaded.


## Cell 16 — Generate `submission.csv`

Generate your final `submission.csv` file for submission.

> Do not modify this cell.

In [18]:
import re, csv, os, time

STREAMLIT_PATTERN = re.compile(
    r"^(https://.+\.streamlit\.app(/.*)?|https?://localhost:8501/?|https?://127\.0\.0\.1:8501/?|https://share\.streamlit\.io/.+)$",
    re.IGNORECASE,
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE,
)

print("=" * 50)
print("Submission Generator")
print("=" * 50)

# Set values directly instead of using input()
streamlit_link = "http://localhost:8501"
langsmith_link = "https://smith.langchain.com/public/ab4bc14d-5dcb-4f25-963a-a81450ecf681/r"

print(f"Streamlit URL received: {streamlit_link}")
print(f"LangSmith URL received: {langsmith_link}")

link_errors = []

if not STREAMLIT_PATTERN.match(streamlit_link):
    link_errors.append("Invalid Streamlit URL.")

if not langsmith_link:
    link_errors.append("LangSmith URL is required.")
elif not LANGSMITH_PATTERN.match(langsmith_link):
    link_errors.append("Invalid LangSmith URL.")

if link_errors:
    print("\n".join(link_errors))
    print("Please provide a real LangSmith shared link before final submission.")

print(f"\nGenerating responses for {len(eval_questions)} questions...\n")

rows = []

for i, q in enumerate(eval_questions):
    qid = q["question_id"]
    question = q["question"]

    try:
        result = ask_bot(question)
        answer = result["answer"]
        status = "OK"
    except Exception as e:
        answer = f"Error: {str(e)}"
        status = "ERROR"

    rows.append({
        "question_id": qid,
        "question_enc": fernet.encrypt(question.encode()).decode(),
        "answer_enc": fernet.encrypt(answer.encode()).decode(),
        "streamlit_link": streamlit_link,
        "langsmith_link": langsmith_link,
    })

    print(f"[{i+1:02d}/{len(eval_questions)}] {qid} ... {status}")

    if i < len(eval_questions) - 1:
        time.sleep(2)

csv_path = "submission.csv"

fieldnames = [
    "question_id",
    "question_enc",
    "answer_enc",
    "streamlit_link",
    "langsmith_link"
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("\nsubmission.csv generated successfully.")

Submission Generator
Streamlit URL received: http://localhost:8501
LangSmith URL received: https://smith.langchain.com/public/ab4bc14d-5dcb-4f25-963a-a81450ecf681/r

Generating responses for 15 questions...

[01/15] Q01 ... OK
[02/15] Q02 ... OK
[03/15] Q03 ... OK
[04/15] Q04 ... OK
[05/15] Q05 ... OK
[06/15] Q06 ... OK
[07/15] Q07 ... OK
[08/15] Q08 ... OK
[09/15] Q09 ... OK
[10/15] Q10 ... OK
[11/15] Q11 ... OK
[12/15] Q12 ... OK
[13/15] Q13 ... OK
[14/15] Q14 ... OK
[15/15] Q15 ... OK

submission.csv generated successfully.


## Cell 17 — Final Checklist

Verify your submission files and links before submitting on Kaggle.

> This cell is pre-filled — just run it.

In [19]:
import re, csv, os

STREAMLIT_PATTERN = re.compile(
    r"^(https://.+\.streamlit\.app(/.*)?|https?://localhost:8501/?|https?://127\.0\.0\.1:8501/?|https://share\.streamlit\.io/.+)$",
    re.IGNORECASE,
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE,
)

print("Final Submission Check")
print("=" * 50)

if os.path.exists("submission.csv"):

    with open("submission.csv", newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    count = len(rows)

    has_fields = all(
        all(
            k in r
            for k in [
                "question_id",
                "question_enc",
                "answer_enc",
                "streamlit_link",
                "langsmith_link"
            ]
        )
        for r in rows
    )

    sl_valid = all(
        STREAMLIT_PATTERN.match(r["streamlit_link"].strip())
        for r in rows
    )

    ll_valid = all(
        LANGSMITH_PATTERN.match(r["langsmith_link"].strip())
        for r in rows
    )

    print(f"submission.csv found ({count} rows)")
    print(f"Required columns present: {has_fields}")
    print(f"Streamlit links valid: {sl_valid}")
    print(f"LangSmith links valid: {ll_valid}")

    if not sl_valid or not ll_valid:
        print("\nPlease regenerate submission.csv with valid links.")

else:
    print("submission.csv not found. Run the previous cell first.")

print("=" * 50)
print("Upload submission.csv to Kaggle to complete your submission.")

Final Submission Check
submission.csv found (15 rows)
Required columns present: True
Streamlit links valid: True
LangSmith links valid: True
Upload submission.csv to Kaggle to complete your submission.
